In [1]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
## define path
basedir = "/Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx"

# output directory
sub_out = "hepatovirus_260113_MSigDB_Hallmark_2020_stat/"
out_folder_path = f"{basedir}/output/250909_4474_samples/final_test/"
out_path = out_folder_path + sub_out
plot_path = out_path + "heatmap_hepatovirus/"

# input file
input_folder_path = out_path
# gene table path
gene_table_path = f"{basedir}/data/gene_selection_by_chatgpt.tsv"
# RNA seq
input_rna_path = f"{basedir}/data/251215_rna_seq/To_kurihara_hepatovirus/output/"

# make directories
if not(os.path.exists(out_folder_path)):
    os.mkdir(out_folder_path)
if not(os.path.exists(out_path)):
    os.mkdir(out_path)
if not(os.path.exists(plot_path)):
    os.mkdir(plot_path)
print("saving files to:", plot_path)

saving files to: /Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/250909_4474_samples/final_test/hepatovirus_260113_MSigDB_Hallmark_2020_stat/heatmap_hepatovirus/


# gene selection

In [3]:
# map term and genes
ttg = defaultdict(list)

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]
        print("\nprocessing...", project)
        print("size:", res.shape)
        print("#NaN:", res.isna().sum().sum())
        print("#uniq Term:", len(set(res["Term"].to_list())))
        
        # term list
        selected_terms = [
            "Interferon Alpha Response",
            "IL-6/JAK/STAT3 Signaling",
            "Xenobiotic Metabolism",
            "Adipogenesis"
        ]

        # map term and lead genes
        for idx in res.index:
            term = res.loc[idx, "Term"]
            if term not in selected_terms:
                continue
            print(term)
            genes = res.loc[idx, "Lead_genes"].split(";")
            if len(genes) != len(set(genes)):
                print("genes not unique")
            for gene in genes:
                if gene not in ttg[term]:
                    ttg[term].append(gene)


processing... PRJNA774885
size: (50, 10)
#NaN: 0
#uniq Term: 50
Interferon Alpha Response
IL-6/JAK/STAT3 Signaling
Xenobiotic Metabolism
Adipogenesis

processing... PRJEB63475
size: (50, 10)
#NaN: 0
#uniq Term: 50
Xenobiotic Metabolism
Adipogenesis
IL-6/JAK/STAT3 Signaling
Interferon Alpha Response


In [4]:
# create table
df = pd.DataFrame(
    [(k, v) for k, vs in ttg.items() for v in vs],
    columns=["term", "gene"]
)
# add group
group = {
    "Interferon Alpha Response": 1,
    "IL-6/JAK/STAT3 Signaling": 2,
    "Xenobiotic Metabolism": 3,
    "Adipogenesis": 4
}
df["group"] = df["term"].apply(lambda x: group[x])
df.to_csv(plot_path + "term2gene.tsv" ,sep="\t")
# show
print(df.shape)
print(df.isna().sum().sum())
df.head(3)

(276, 3)
0


,term,gene,group
0,Interferon Alpha Response,OGFR,1
1,Interferon Alpha Response,EPSTI1,1
2,Interferon Alpha Response,DHX58,1


# Creates table for heatmap

In [5]:
## show gene df
genedf = pd.read_table(gene_table_path, skiprows=1)
genedf

,group,方向,term,gene,解析での解釈（超要約）
0,1,上昇,Interferon Alpha Response,IFIH1,dsRNAセンサー(MDA5)。ウイルスRNA検知→I型IFN/ISG誘導の起点 (Scie...
1,1,上昇,Interferon Alpha Response,IRF7,I型IFN誘導の主要転写因子（マスター制御） (PubMed)
2,1,上昇,Interferon Alpha Response,EIF2AK2,PKR。dsRNAで活性化し翻訳抑制などの抗ウイルス機構 (Frontiers)
3,1,上昇,Interferon Alpha Response,RSAD2,Viperin。代表的ISGで広範な抗ウイルス作用 (PubMed)
4,1,上昇,Interferon Alpha Response,TRIM25,抗ウイルス自然免疫（RNAセンサー経路）を増強するE3 ligase (PMC)
5,2,上昇,IL-6/JAK/STAT3 Signaling,STAT3,IL-6などで活性化する転写因子。肝炎/肝障害・炎症応答で頻出 (PMC)
6,2,上昇,IL-6/JAK/STAT3 Signaling,IL6ST,gp130（共通受容体サブユニット）。IL-6→JAK/STAT3の中核 (PubMed)
7,2,上昇,IL-6/JAK/STAT3 Signaling,SOCS1,JAK/STATの負のフィードバック（過剰なサイトカイン/IFNシグナルを抑制） (Nature)
8,2,上昇,IL-6/JAK/STAT3 Signaling,MYD88,TLR/IL-1Rアダプター。自然免疫炎症性サイトカイン誘導の要 (Nature)
9,2,上昇,IL-6/JAK/STAT3 Signaling,CD14,TLRの共受容体。PAMP/DAMP認識と自然免疫活性化に関与 (PMC)


In [6]:
# check if all genes exist
# map and term and genes
test_dic = defaultdict(set)
all_genes = set()

# read gene table
genedf = pd.read_table(gene_table_path, skiprows=1)
genes_oi = []
for g in genedf["gene"].to_list():
    if g not in genes_oi:
        genes_oi.append(g)
print("# uniq gene oi:", len(set(genes_oi)))
print("size of gene df:", genedf.shape)
t_order = []
for t in genedf["term"].to_list():
    if t not in t_order:
        t_order.append(t)

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # only selected term
        for idx in res.index:
            term = res.loc[idx, "Term"]
            if term not in selected_terms:
                continue
            genes = res.loc[idx, "Lead_genes"].split(";")

            # get gene info
            for gene in genes:
                all_genes.add(gene)
                 # get term info per gene
                if gene not in genes_oi:
                    continue
                test_dic[gene].add(term)

# check if genes in chatGPT table exist in table
print(f"\n#all genes: {len(all_genes)}")
for goi in genes_oi:
    if goi not in sorted(list(all_genes)):
        print(f"### {goi} not exist! ###")
# - Yes

# check if genes have uniq term
testdf = pd.DataFrame(
    [(k, v) for k, vs in test_dic.items() for v in sorted(list(vs))],
    columns=["gene", "term"]
)
print(test_dic)
# - Yes

# term anno df
term_bin = (pd.crosstab(testdf["gene"], testdf["term"]) > 0).astype(int)
term_bin = term_bin[t_order]
term_bin.columns = term_bin.columns.map(lambda x: x.replace("-", "_"))
# sort
term_bin = term_bin.sort_values(
            "gene",
            key=lambda s: pd.Categorical(s, categories=genes_oi, ordered=True)
        )
# save
term_bin.to_csv(plot_path + "term_anno.csv")

# show
term_bin.head(3)

# uniq gene oi: 21
size of gene df: (21, 5)

#all genes: 260
defaultdict(<class 'set'>, {'IFIH1': {'Interferon Alpha Response'}, 'EIF2AK2': {'Interferon Alpha Response'}, 'RSAD2': {'Interferon Alpha Response'}, 'TRIM25': {'Interferon Alpha Response'}, 'MYD88': {'IL-6/JAK/STAT3 Signaling'}, 'SOCS1': {'IL-6/JAK/STAT3 Signaling'}, 'IL6ST': {'IL-6/JAK/STAT3 Signaling'}, 'STAT3': {'IL-6/JAK/STAT3 Signaling'}, 'ABCC2': {'Xenobiotic Metabolism'}, 'ACOX1': {'Adipogenesis', 'Xenobiotic Metabolism'}, 'ALDH2': {'Adipogenesis', 'Xenobiotic Metabolism'}, 'LPL': {'Adipogenesis'}, 'PLIN2': {'Adipogenesis'}, 'LIPE': {'Adipogenesis'}, 'ADIPOQ': {'Adipogenesis'}, 'ADIPOR2': {'Adipogenesis'}, 'NQO1': {'Xenobiotic Metabolism'}, 'CYP27A1': {'Xenobiotic Metabolism'}, 'CIDEA': {'Adipogenesis'}, 'CD14': {'IL-6/JAK/STAT3 Signaling'}, 'IRF7': {'Interferon Alpha Response'}})


term,Interferon Alpha Response,IL_6/JAK/STAT3 Signaling,Xenobiotic Metabolism,Adipogenesis
gene,,,,
IFIH1,1,0,0,0
IRF7,1,0,0,0
EIF2AK2,1,0,0,0


In [7]:
# obtain pivot table
dfs = []
for dirpath, dirnames, filenames in os.walk(input_rna_path):
    for fname in filenames:
        if not(fname.startswith("result")):
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)

        # extract genes
        df_gene = project.loc[project["gene_name_hs"].isin(genes_oi), ["gene_name_hs", "log2FoldChange"]].reset_index(drop=True).copy()
        df_gene["Project"] = proname
        print(f"size for {proname}: {df_gene.shape}")

        # append
        dfs.append(df_gene)

# concat
df_cat = pd.concat(dfs)
print("\nsize for all:", df_cat.shape)
print(df_cat.head(3))
# pivot
df_pivot = (df_cat.pivot_table(index="gene_name_hs", columns="Project", values="log2FoldChange").sort_index())
print("\nsize for df pivot:", df_pivot.shape)
print("#NaN for df pivot:", df_pivot.isna().sum().sum())
# order
df_pivor_sorted = df_pivot.sort_values(
            "gene_name_hs",
            key=lambda s: pd.Categorical(s, categories=genes_oi, ordered=True)
        )
df_pivor_sorted = df_pivor_sorted[["PRJEB63475", "PRJNA774885"]]
print("size for sorted:", df_pivor_sorted.shape)
# save
df_pivor_sorted.to_csv(plot_path + "gene_project_log2FC_matrix.csv")
# show
df_pivor_sorted.head(3)


processing... PRJNA774885
size for PRJNA774885: (21, 3)

processing... PRJEB63475
size for PRJEB63475: (21, 3)

size for all: (42, 3)
  gene_name_hs  log2FoldChange      Project
0      ADIPOR2       -0.271052  PRJNA774885
1        MYD88        1.389870  PRJNA774885
2        CIDEA        0.534404  PRJNA774885

size for df pivot: (21, 2)
#NaN for df pivot: 0
size for sorted: (21, 2)


Project,PRJEB63475,PRJNA774885
gene_name_hs,,
IFIH1,0.422660,1.872193
IRF7,2.006851,0.648652
EIF2AK2,0.597847,1.795255
